### Tools

Models can request to call tools that performs task such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    2. A function or coroutine to execute.

In [3]:
import os
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3.6-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Question**: The user asks "Why do parrots talk?" This is a common question about animal behavior, specifically avian vocalization and speech mimicry.\n\n2.  **Identify Key Concepts**:\n   - Parrots don\'t actually "talk" in the human sense (they don\'t understand language semantically in the same way humans do)\n   - They mimic sounds, including human speech\n   - Evolutionary/biological reasons for vocal mimicry\n   - Social/behavioral functions\n   - Physical anatomy enabling speech-like sounds\n   - Cognitive aspects\n\n3.  **Structure the Response**:\n   - Clarify the misconception (they mimic, not truly "talk")\n   - Explain the biological/anatomical basis\n   - Discuss evolutionary/social reasons\n   - Mention cognitive aspects\n   - Provide examples/context\n   - Keep it clear, scientific, and accessible\n\n4.  **Draft - Section by Section**:\n   *(Introduction)* Parrots don’t actually “talk” the 

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""   
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response=model_with_tools.invoke("Whats the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify the user\'s intent: Get weather information for Boston.\n2.  Identify available tools: `get_weather` function takes a `location` parameter.\n3.  Extract parameters: location = "Boston".\n4.  Call the function: `get_weather(location="Boston")`.\n5.  Formulate response based on the function\'s output. (Will wait for tool output) -> Actually, I just generate the tool call. I am an AI, I will output the tool call.\n\nWait, I should just generate the tool call directly.\nParameters: `{"location": "Boston"}`.\nCheck constraints: function requires `location` (string). All good.\nProceed. \nOutput matches tool call format.✅\n', 'tool_calls': [{'id': 'nrb7rqbxj', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 189, 'prompt_tokens': 274, 'total_tokens': 463, 'completion_time': 0.370029348, 'completion_tokens_d

### Tool Execution Loops

In [6]:
#Step 1: Model generates tool calls
messages =[{"role":"user","content":"Whats the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with geneated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's sunny in Boston.


In [7]:
messages

[{'role': 'user', 'content': 'Whats the weather in Boston?'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  Identify user intent: User wants to know the weather in Boston.\n2.  Identify required tool: `get_weather`\n3.  Identify required parameters: `location` = "Boston"\n4.  Call the tool.\n5.  Format the response based on the tool\'s output. (Will wait for tool output, but I can just call it now).\n\nTool call: `get_weather(location="Boston")`\nProceed. \nNo extra thinking needed. All parameters are available. Output matches schema.✅\nLet\'s generate the tool call. \nWait, I should just output the tool call directly.\nDone. \nProceeding. \n[Self-Correction/Verification]\n- Function name: `get_weather`\n- Parameter: `location` (string) -> "Boston"\n- Matches schema perfectly.\n- Ready. \nOutput matches response format.✅\n', 'tool_calls': [{'id': 'xynfh1g83', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'fun

### Tool to fetch current weather condition for any location using Weatherstack API

In [8]:
import os
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv()

@tool
def get_weather(location: str) -> str:
    """Fetch current real-time weather information for a given city or location using Weatherstack."""
    api_key = os.getenv("WEATHERSTACK_API_KEY")
    if not api_key:
        return "Error: WEATHERSTACK_API_KEY is not set in environment variables."

    # Weatherstack free tier uses HTTP endpoint
    url = "http://api.weatherstack.com/current"
    params = {
        "access_key": api_key,
        "query": location,
        "units": "m"  # 'm' for Metric (Celsius), 'f' for Fahrenheit
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()

        if "error" in data:
            return f"Weatherstack API Error: {data['error'].get('info', 'Unknown error')}"

        current = data.get("current", {})
        loc = data.get("location", {})

        city = loc.get("name", location)
        country = loc.get("country", "")
        temp = current.get("temperature")
        feels_like = current.get("feelslike")
        weather_desc = ", ".join(current.get("weather_descriptions", []))
        humidity = current.get("humidity")
        wind_speed = current.get("wind_speed")

        return (
            f"Current Weather in {city}, {country}:\n"
            f"- Condition: {weather_desc}\n"
            f"- Temperature: {temp}°C (Feels like {feels_like}°C)\n"
            f"- Humidity: {humidity}%\n"
            f"- Wind Speed: {wind_speed} km/h"
        )
    except Exception as e:
        return f"Error fetching weather data: {str(e)}"


# Direct tool test
result = get_weather.invoke({"location": "Mumbai"})
print(result)


Current Weather in Mumbai, India:
- Condition: Patchy rain nearby
- Temperature: 28°C (Feels like 32°C)
- Humidity: 78%
- Wind Speed: 23 km/h


In [9]:
from langchain.agents import create_agent

# Create an agent equipped with your real-time weather tool
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",  # or "openai:gpt-4o-mini", "groq:llama-3.3-70b-versatile"
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use the weather tool whenever asked about current weather conditions."
)

# Run the agent
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the weather currently like in Mumbai, and should I carry an umbrella?"}]
})

print(response["messages"][-1].content)


[{'type': 'text', 'text': "The current weather in Mumbai is 28°C with patchy rain nearby and high humidity (78%). \n\nYes, you should definitely **carry an umbrella** with you just in case you run into any sudden showers while you're out!", 'extras': {'signature': 'El4KXAERTTIPuwdE1sRBlCGkSQMVUZs5psysV60c9VP4FyeAZ1wyB9jZ99R7kRkRX/QPQ5aAkmKSVMHmCSbeVNHt0k5fpbroNP2hxO6kqCWNe3IxtaFfnFBqd0oxlPUm'}}]
